# 12.13 · 向量数据库 / Vector Databases

> **课程定位 / Where this fits**
> 第 13 课，**Part 12**。RAG(12.12)和一切语义检索的"引擎室"。
> Lesson 13, **Part 12**. The "engine room" of RAG (12.12) and all semantic search.
>
> RAG 要在**几百万甚至上亿个向量**里，为每个查询找出最相似的 top-k。**暴力比较**(挨个算相似度)在大规模下太慢——每次查询都要扫全库。**向量数据库**(FAISS / Chroma / Pinecone / Milvus)用**近似最近邻(ANN)** 索引，用"**牺牲一点点准确率换取上百倍速度**"实现毫秒级检索。本课**从零实现**暴力检索和一个**基于聚类的 ANN 索引(IVF)**，亲手测出"速度 vs 召回率"的权衡——这正是向量数据库的核心。
> RAG must find the top-k most similar among **millions or billions of vectors** per query. **Brute force** (compute similarity to every vector) is too slow at scale — each query scans the whole store. **Vector databases** (FAISS/Chroma/Pinecone/Milvus) use **approximate nearest neighbor (ANN)** indexes that "**trade a little accuracy for 100× speed**," achieving millisecond search. We **implement from scratch** brute-force search and a **cluster-based ANN index (IVF)**, measuring the "speed vs recall" trade-off — the heart of vector DBs.
>
> 💼 **实战/面试视角**："为什么需要ANN / 暴力检索复杂度 / HNSW/IVF/PQ 原理 / 召回率vs速度权衡" 是检索/RAG 系统岗常考。
> 💼 **Practical/interview angle:** "why ANN / brute-force complexity / HNSW/IVF/PQ / recall vs speed" — retrieval/RAG-systems questions.

> 📐 **符号约定 / Notation**
> - ANN —— 近似最近邻(approximate nearest neighbor) / approximate nearest neighbor
> - 召回率(recall@k) —— ANN 找到的 top-k 与真实 top-k 的重合比例 / overlap with the exact top-k

> 💡 **面试相关 / Interview-relevant**
> - "为什么不能暴力检索(复杂度)"（出镜率 ★★★★）
> - "ANN 的核心思想(牺牲精度换速度)"（★★★★★）
> - "HNSW / IVF / PQ 的区别"（★★★★）
> - "召回率与速度/内存的权衡"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解为什么大规模检索需要 ANN(暴力的复杂度)。
   Understand why large-scale retrieval needs ANN (brute-force complexity).
2. **从零实现暴力 kNN 检索**(精确基线)。
   Implement brute-force kNN from scratch (the exact baseline).
3. **从零实现 IVF(聚类)ANN 索引**。
   Implement an IVF (cluster-based) ANN index from scratch.
4. 实测"速度 vs 召回率"权衡, 了解 HNSW/PQ。
   Measure the speed-vs-recall trade-off; know HNSW/PQ.

## 目录 / TOC
1. [为什么需要 ANN ⭐](#1)
2. [暴力检索(精确基线，从零)⭐](#2)
3. [IVF：聚类加速检索（从零）⭐](#3)
4. [速度 vs 召回 + HNSW/PQ + 小结 ⭐](#4)


<a id="1"></a>
## 1. 为什么需要 ANN ⭐ / Why ANN

**精确最近邻**：给一个查询向量，要找最相似的 top-k，最直接的办法是**和库里每一个向量都算一次相似度**，再排序取前 k。这叫**暴力检索(brute-force / flat)**，结果**100% 准确**。
**Exact nearest neighbor:** given a query, the simplest way to find top-k is to **compute similarity to every vector** in the store, then sort. This **brute-force (flat)** search is **100% accurate**.

问题是**复杂度 $O(N \times d)$**：库里有 $N$ 个 $d$ 维向量，每次查询都要 $N$ 次向量运算。当 $N$ 是百万、亿级，且要支撑高 QPS 时，**太慢、太贵**。
The problem is **$O(N \times d)$ complexity**: $N$ vectors of dim $d$, each query does $N$ vector ops. At millions/billions of vectors and high QPS, **too slow, too costly**.

**ANN(近似最近邻)** 的核心思想(面试要点)：**不保证找到绝对最近的，只要以极高概率找到足够近的就行**——用**牺牲一点点召回率**换取**几十上百倍的速度**。绝大多数应用(如 RAG)完全能接受 95%+ 的召回率换来的巨大加速。
**ANN's core idea** (interview): **don't guarantee the exact nearest, just find good-enough ones with high probability** — trade **a little recall** for **tens-to-hundreds× speed**. Most apps (e.g. RAG) happily accept 95%+ recall for the huge speedup.


<a id="2"></a>
## 2. 暴力检索(精确基线，从零)⭐ / Brute-Force Search (Exact Baseline)

先实现暴力检索作为**精确基线**(100% 召回，但慢)。我们造一个 5 万向量的库。
First, brute-force search as the **exact baseline** (100% recall, but slow). We build a store of 50k vectors.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, time
sns.set_theme(style="whitegrid"); np.random.seed(0)

N, d = 50000, 64                                          # 5万个 64维向量 / 50k vectors of dim 64
# 造"有聚类结构"的数据(模拟真实嵌入: 相似内容自然聚成簇); 纯随机数据无结构, 是ANN最坏情况
# clustered data (mimics real embeddings: similar items naturally cluster); pure random has no structure (worst case)
n_true = 60
centers = np.random.randn(n_true, d).astype(np.float32)
assign = np.random.randint(n_true, size=N)
db = centers[assign] + 1.3 * np.random.randn(N, d).astype(np.float32)   # 围绕中心的簇(有重叠, 更真实) / overlapping blobs
db /= np.linalg.norm(db, axis=1, keepdims=True)           # 归一化 → 余弦相似度=点积 / normalize so cosine=dot
qi = np.random.choice(N, 100, replace=False)              # 查询=库中某点的轻微扰动(有真实近邻) / queries near data
queries = db[qi] + 0.1 * np.random.randn(100, d).astype(np.float32)
queries /= np.linalg.norm(queries, axis=1, keepdims=True)

def brute_force(q, k=10):
    sims = db @ q                                         # 和库里每个向量算点积(余弦) / dot with every vector
    return set(np.argsort(sims)[::-1][:k])                # 取 top-k / top-k indices

t0 = time.time()
exact = [brute_force(q) for q in queries]                 # 精确答案(作为召回率的"标准答案") / ground truth
bf_time = time.time() - t0
print(f"暴力检索: {N} 向量, {len(queries)} 次查询, 共 {bf_time*1000:.0f} ms ({bf_time/len(queries)*1000:.2f} ms/查询)")
print(f"复杂度 O(N×d): 每次查询都要和全部 {N} 个向量算相似度 → N 越大越慢")
print("结果 100% 精确, 但大规模(百万/亿)下太慢 → 需要 ANN 加速")


<a id="3"></a>
## 3. IVF：聚类加速检索（从零）⭐ / IVF: Cluster-Based Search

**IVF(倒排文件索引)** 是最直观的 ANN 之一，思想很简单：**先把所有向量聚成若干个簇(用 k-means)；查询时，只在离查询最近的几个簇里搜索**，而不是全库。
**IVF (Inverted File index)** is one of the most intuitive ANN methods: **cluster all vectors into groups (k-means); at query time, search only the few clusters nearest the query**, not the whole store.

直觉：相似的向量大概率落在同一个簇里。所以只看"查询所在区域附近的几个簇"，就能找到绝大多数真正的近邻——但只搜了**一小部分**向量。**nprobe** = 搜索的簇数：nprobe 越大→召回越高但越慢，越小→越快但可能漏掉。这就是那个**速度↔召回**的旋钮。
Intuition: similar vectors likely fall in the same cluster. Searching just "the few clusters near the query" finds most true neighbors while scanning only a **fraction** of vectors. **nprobe** = number of clusters searched: larger → higher recall but slower; smaller → faster but may miss. That's the **speed↔recall** knob.


In [ ]:
from sklearn.cluster import KMeans
n_clusters = 200
t0 = time.time()
km = KMeans(n_clusters=n_clusters, n_init=3, random_state=0).fit(db)   # 离线: 把库聚成200个簇 / cluster offline
centroids = km.cluster_centers_.astype(np.float32)
# 建"倒排表": 每个簇 → 属于它的向量索引 / inverted lists: cluster → member vector indices
inverted = [np.where(km.labels_ == c)[0] for c in range(n_clusters)]
print(f"建索引(k-means {n_clusters}簇) 耗时 {time.time()-t0:.1f}s (离线一次性)")

def ivf_search(q, k=10, nprobe=8):
    cl = np.argsort(centroids @ q)[::-1][:nprobe]         # 找最近的 nprobe 个簇中心 / nearest nprobe centroids
    cand = np.concatenate([inverted[c] for c in cl])      # 只取这些簇里的候选向量 / candidates from those clusters
    sims = db[cand] @ q                                   # 只对候选算相似度(远少于N) / score only candidates
    return set(cand[np.argsort(sims)[::-1][:k]])

# 测召回率: ANN 找到的 top-10 与精确 top-10 的重合 / recall@10 vs exact
def recall(nprobe):
    hit = total = 0; t0 = time.time()
    for q, ex in zip(queries, exact):
        got = ivf_search(q, nprobe=nprobe); hit += len(got & ex); total += len(ex)
    return hit/total, (time.time()-t0)/len(queries)*1000
r, ms = recall(nprobe=8)
print(f"IVF (nprobe=8): 召回率@10 = {r:.3f}, {ms:.2f} ms/查询  (vs 暴力 {bf_time/len(queries)*1000:.2f} ms, 提速 ~{(bf_time/len(queries)*1000)/ms:.0f}x)")
print("只搜8/200个簇 → 候选向量少很多 → 快很多, 召回率仍很高(相似向量多在同簇)")


<a id="4"></a>
## 4. 速度 vs 召回 + HNSW/PQ + 小结 ⭐ / Speed vs Recall & HNSW/PQ

ANN 的本质是个**旋钮**：nprobe 越大，搜的簇越多 → 召回率越高，但越慢。下面画出这条权衡曲线。
ANN is fundamentally a **knob**: larger nprobe → more clusters searched → higher recall but slower. Let's plot this trade-off.


In [ ]:
nprobes = [1, 2, 4, 8, 16, 32, 64]
recalls, times = [], []
for npb in nprobes:
    r, ms = recall(npb); recalls.append(r); times.append(ms)
fig, ax1 = plt.subplots(figsize=(8,4.5))
ax1.plot(nprobes, recalls, "o-", color="#39c", label="召回率@10")
ax1.set_xlabel("nprobe (搜索的簇数)"); ax1.set_ylabel("召回率@10", color="#39c"); ax1.set_xscale("log", base=2)
ax2 = ax1.twinx(); ax2.plot(nprobes, times, "s--", color="#e67", label="ms/查询")
ax2.axhline(bf_time/len(queries)*1000, color="gray", ls=":", label="暴力检索时间")
ax2.set_ylabel("ms/查询", color="#e67")
ax1.set_title("ANN 速度↔召回 权衡: nprobe 越大召回越高但越慢"); fig.legend(loc="center right")
plt.tight_layout(); plt.show()
for npb, r, ms in zip(nprobes, recalls, times):
    print(f"  nprobe={npb:2}: 召回率 {r:.3f}, {ms:.2f} ms/查询")
print("\n旋钮: nprobe 小→极快但漏召回; 大→召回高但接近暴力; 实战取一个'召回够用且够快'的点")


**主流 ANN 算法**(面试常让对比)：
**Mainstream ANN algorithms** (interviewers love comparisons):
- **IVF(倒排/聚类)**：本课实现的。先聚类、只搜近簇。简单有效。
  **IVF (cluster):** what we built — cluster, search near clusters. Simple, effective.
- **HNSW(分层可导航小世界图)**：把向量连成**多层图**，查询时从稀疏的高层"跳"到密集的低层、贪心地沿边走向最近邻。**召回高、速度快**，是当今最流行的 ANN 索引(很多向量库默认)。
  **HNSW (Hierarchical Navigable Small World):** a **multi-layer graph** of vectors; queries hop from sparse top layers to dense bottom, greedily walking edges toward neighbors. **High recall, fast** — the most popular ANN index today (default in many vector DBs).
- **PQ(乘积量化)**：把高维向量**压缩**成短编码(省内存、加速距离计算)。常和 IVF 组合(IVF-PQ)处理十亿级。
  **PQ (Product Quantization):** **compress** vectors into short codes (save memory, speed distance). Often combined as IVF-PQ for billion-scale.

**向量数据库产品**：FAISS(库, Meta)、Chroma/Qdrant/Milvus(开源)、Pinecone/Weaviate(托管)。它们封装了这些索引 + 增删改查 + 元数据过滤 + 持久化 + 分布式。
**Vector DB products:** FAISS (library, Meta), Chroma/Qdrant/Milvus (open-source), Pinecone/Weaviate (managed). They wrap these indexes + CRUD + metadata filtering + persistence + distribution.

```
暴力检索: 和全部N个向量算相似度, O(N×d), 100%精确但大规模太慢
ANN核心: 牺牲一点召回换几十上百倍速度; 用旋钮(nprobe等)调权衡
IVF: k-means聚类→只搜最近nprobe个簇; nprobe大=召回高但慢, 小=快但漏
HNSW: 多层图导航, 召回高速度快, 当今最流行; PQ: 量化压缩省内存(IVF-PQ处理十亿级)
召回率@k: ANN找到的top-k与精确top-k的重合; 实战取'够用且够快'的点
产品: FAISS/Chroma/Qdrant/Milvus/Pinecone; 封装索引+CRUD+元数据过滤+持久化
```

### 💡 面试速查 / Interview cheat-sheet
1. **为何ANN**: 暴力O(N×d)在百万/亿向量太慢; ANN牺牲精度换速度。
   Why ANN: brute-force O(N×d) too slow at scale; ANN trades accuracy for speed.
2. **IVF**: 聚类+只搜近簇; nprobe 调速度↔召回。
   IVF: cluster + search near clusters; nprobe tunes speed↔recall.
3. **HNSW**: 多层图导航, 高召回快速, 最流行。
   HNSW: multi-layer graph navigation, high recall, fast, most popular.
4. **PQ**: 量化压缩向量省内存; IVF-PQ 处理十亿级。
   PQ: quantize/compress to save memory; IVF-PQ for billions.
5. **召回vs速度**: ANN 的核心权衡, 取够用的点。
   Recall vs speed: ANN's core trade-off; pick a good-enough point.

### 下一节 / Next
**12.14 LLM 智能体(Agents)**——让 LLM 不只"回答", 而是**自主使用工具、多步规划、完成任务**。基于 ReAct 思想, Agent 能调用搜索/计算器/API/代码执行, 把 LLM 变成"会做事的助手"。我们会**从零搭一个工具调用 Agent 循环**。
**12.14 LLM Agents** — make an LLM not just "answer" but **autonomously use tools, plan multi-step, and complete tasks**. Built on ReAct, agents call search/calculator/APIs/code, turning the LLM into a "doer." We'll build a tool-using agent loop from scratch.
